# Laya: does more data help? + fixing the boundary question (Kaggle, GPU T4 ×2)

Second run on `laya_v1`, after `laya-wahy-finetuning-finished.ipynb`. It answers two questions in one background commit:

**1. Learning curve: is more data worth generating?** The same recipe (soft labels, data as is) trained on 25 / 50 / 75 / 100 %
of the training lectures. If the test score still climbs from 75 % to 100 %, more data will help; if it is flat, generating more of
the same will not, and the effort should go to other fixes (hard cases, the errors, the boundary).

**2. The boundary question collapsed** in the first run: every model answered `COMPLETE_SPAN` for every quote. Why: 81 % of the
training boundary labels are `COMPLETE_SPAN` (Sonnet marks 84–87 % of its quotes complete), so "always complete" is the cheapest
answer, and on val (86 % complete) it even scores 86 % accuracy, which hid the problem. The runs `bnd_x3` / `bnd_x6` repeat every
*cut-off* boundary item 3× / 6× in training, so the model can no longer win by ignoring them. Verdict items are untouched.

| run | training data | boundary repeats | question it answers |
|---|---|---|---|
| `curve_100` | all 57 training lectures (2,863 rows) | – | the reference: soft_asis at 2 epochs |
| `bnd_x3` | same | cut-off items ×3 (42 % of boundary items) | does oversampling fix the boundary? |
| `bnd_x6` | same | cut-off items ×6 (59 %) | more of it: better, or now too eager? |
| `curve_25` / `_50` / `_75` | ¼, ½, ¾ of the lectures (nested) | – | the learning curve |

What changed vs the first run: **2 epochs** instead of 4 (val loss was lowest at epoch 2), the boundary is scored with
**balanced accuracy** (mean recall of the 4 classes: "always complete" = 0.25, not 0.86), and a **no-model word-overlap rule**
is scored next to the models as the bar to beat. Val and test are the same files as before, so the numbers compare directly.

### Kaggle settings (right-hand panel)
* **Accelerator:** `GPU T4 x2` · **Internet:** `On` · **Secrets:** `HF_TOKEN` (only if `PUSH_TO_HUB`)
* **Input:** the `laya_v1` dataset, **new version** with the updated `train_ddp.py` / `eval_laya.py` (section 3 checks it)
* **Run:** *Save Version → Save & Run All (Commit)*, browser can be closed. Expect **~1 h 45 min** (6 trainings, 6 evaluations).
  The order puts `curve_100`, `bnd_x3`, `bnd_x6` first, so a session that dies late still has the boundary answer.

## 1. Environment & dual T4 GPU check

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}! Notebook options -> Accelerator -> GPU T4 x2."
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Install dependencies

In [ ]:
# pinned to the 0.4 line the Wahy data was validated with (laya.train / laya.calibrate are identical 0.4.12 -> 0.4.15)
!pip install -q -U "laya>=0.4.12,<0.5" "transformers>=4.48.0" safetensors huggingface_hub pandas matplotlib tabulate
import laya, transformers
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)

## 3. Dataset, scripts and the base checkpoint
Stops early if the dataset is still the first version: this run needs `--boundary-oversample` in `train_ddp.py` and the
boundary rule in `eval_laya.py`.

In [ ]:
import glob, json, sys
from pathlib import Path
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config

DATA = next(Path(p).parent for p in sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
            if json.loads(Path(p).read_text()).get("name") == "laya_v1")
assert "--boundary-oversample" in (DATA / "train_ddp.py").read_text() and "def rule_boundary" in (DATA / "eval_laya.py").read_text(), (
    "Old scripts in the dataset: upload the rebuilt laya_v1.zip as a NEW VERSION of the dataset (see the guide), then re-run.")

manifest = json.loads((DATA / "manifest.json").read_text())
sys.path.insert(0, str(DATA))
import eval_laya

print("dataset:", DATA, "| created", manifest["created"], "| base model", manifest["base_model"])
MODEL_ID = manifest["base_model"]          # convaiinnovations/laya-multilingual
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
print("base checkpoint:", model_dir, json.loads(Path(model_dir, "rl_agent_config.json").read_text())["encoder"])

## 4. Settings
`RUNS` runs in this order; a failing run is logged and the next one still runs. Everything else is the first run's recipe
(effective batch 64, RLCD, the same learning rates), so the only differences are the ones in the table above.

In [ ]:
RUNS = {   # name -> (training subset, boundary oversample)
    "curve_100": ("curve_100", 1),
    "bnd_x3":    ("curve_100", 3),
    "bnd_x6":    ("curve_100", 6),
    "curve_25":  ("curve_25", 1),
    "curve_50":  ("curve_50", 1),
    "curve_75":  ("curve_75", 1),
}
TRAIN_ARGS = "--epochs 2 --micro-batch 8 --grad-accum 4 --lr-encoder 2.5e-5 --lr-head 1e-4 --loss rlcd"
APP_FALSE_SHARE = 0.10      # the app's real stream is ~10 % "not a quote" (prior-corrected numbers)
SEED = 20261026             # which lectures go into the 25 / 50 / 75 % subsets
KEEP_WEIGHTS = {"curve_100", "bnd_x3", "bnd_x6"}   # the curve_25/50/75 weights are deleted after evaluation (~1.2 GB each)
PUSH_TO_HUB = True          # upload the KEEP_WEIGHTS models to PRIVATE Hub repos (needs the HF_TOKEN secret)
HUB_PREFIX = "ZenoAbderrahim/laya-wahy-lc"          # repo = f"{HUB_PREFIX}-{run}"
# first run (laya-wahy-finetuning-finished.ipynb), soft_asis at 4 epochs: the dashed reference line in the plots
V1_SOFT_ASIS = {"you · accuracy": 0.914, "you · AUC": 0.944, "you · real wrongly rejected": 0.064,
                "you + Sonnet · AUC": 0.960, "you + Sonnet · accuracy": 0.909, "you · boundary balanced acc": 0.25}

WORK = Path("/kaggle/working")
for d in ("results", "logs", "data"):
    (WORK / d).mkdir(parents=True, exist_ok=True)

def run(cmd, log):
    # Run a shell command, stream its output here and into a log file; returns the exit code.
    print("$", cmd, flush=True)
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line)
            if "Warning" not in line and "warn(" not in line:
                print(line, end="", flush=True)
        return p.wait()

## 5. The 25 / 50 / 75 % subsets
Whole lectures, as in the split (a lecture's findings are near-copies of each other). **Nested**: the 50 % lectures contain the
25 % ones and so on, so each step only *adds* data. **Stratified**: the 9 review lessons (the only ones with your labels) and the
48 batch lectures are subsampled separately, so every subset has the same mix of your labels and Sonnet's. Val and test do not
change.

In [ ]:
import random
from collections import defaultdict

train_rows = eval_laya.read_jsonl(DATA / "soft_asis" / "train.jsonl")
docs = defaultdict(list)
for r in train_rows:
    docs[(r["meta"]["group"], r["meta"]["doc_id"])].append(r)
by_group = defaultdict(list)
for g, d in sorted(docs):
    by_group[g].append(d)
rng = random.Random(SEED)
for g in by_group:
    rng.shuffle(by_group[g])        # one order per group; a fraction takes its first lectures, so the subsets are nested

def describe(rows):
    b = [r["meta"]["boundary"] for r in rows if r["meta"]["boundary"]]
    return {"lectures": len({r["meta"]["doc_id"] for r in rows}), "rows": len(rows),
            "your labels": sum(r["meta"]["source"] == "user" for r in rows),
            "not a quote": round(sum(r["meta"]["verdict"] == "FALSE" for r in rows) / len(rows), 4),
            "boundary items": len(b), "cut-off boundary items": sum(x != "COMPLETE_SPAN" for x in b)}

subsets, false_share = {}, {}
for pct in (25, 50, 75, 100):
    keep = {d for g, ds in by_group.items() for d in ds[:max(1, round(len(ds) * pct / 100))]}
    rows = [r for r in train_rows if r["meta"]["doc_id"] in keep]
    path = WORK / "data" / f"curve_{pct}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
    subsets[f"curve_{pct}"] = describe(rows)
    false_share[f"curve_{pct}"] = subsets[f"curve_{pct}"]["not a quote"]

import pandas as pd
pd.DataFrame(subsets).T

## 6. Train + evaluate
Per run: `torchrun` trains on both GPUs, then `eval_laya.py` scores the model on the full test set. The val line after each
epoch now also prints the boundary's **balanced** accuracy: it should climb above 0.25 for the `bnd_*` runs.

In [ ]:
import time
status = {}
for name, (subset, oversample) in RUNS.items():
    out = WORK / "models" / name
    t0 = time.time()
    try:
        rc = run(f"torchrun --standalone --nproc_per_node=2 {DATA}/train_ddp.py {model_dir} {out} "
                 f"--train {WORK}/data/{subset}.jsonl --val {DATA}/soft_asis/val.jsonl --name laya-wahy-{name} "
                 f"--boundary-oversample {oversample} {TRAIN_ARGS}", WORK / "logs" / f"train_{name}.log")
        if rc != 0 or not (out / "model.safetensors").exists():
            status[name] = f"training failed (exit {rc}), see logs/train_{name}.log"
            continue
        baselines = f"--baselines {DATA}/baselines.json" if name == "curve_100" else ""   # JEV / Gemini lines, once
        rc = run(f"python {DATA}/eval_laya.py predict --model {out} --name {name} --test {DATA}/test.jsonl "
                 f"--out {WORK}/results/{name}.json --train-false-share {false_share[subset]} "
                 f"--app-false-share {APP_FALSE_SHARE} --batch-size 32 {baselines}", WORK / "logs" / f"eval_{name}.log")
        status[name] = f"ok in {(time.time() - t0) / 60:.0f} min" if rc == 0 else f"evaluation failed (exit {rc})"
        if rc == 0 and name not in KEEP_WEIGHTS:
            (out / "model.safetensors").unlink()          # keep train_log.json and the config, drop the weights
    except Exception as e:
        status[name] = f"error: {e}"
    print(f"\n##### {name}: {status[name]}\n", flush=True)
json.dump(status, open(WORK / "results" / "status.json", "w"), indent=1)
status

## 7. Load the results (+ the two no-model boundary baselines)

In [ ]:
from IPython.display import Markdown, display
test_rows = eval_laya.read_jsonl(DATA / "test.jsonl")
res = {n: json.load(open(WORK / "results" / f"{n}.json")) for n in RUNS if (WORK / "results" / f"{n}.json").exists()}
always_complete = {r["id"]: {"boundary_status": {b: float(b == "COMPLETE_SPAN") for b in eval_laya.BOUNDARIES}} for r in test_rows}
boundary_baselines = [eval_laya.evaluate(test_rows, always_complete, "always COMPLETE_SPAN (= the first run)"),
                      eval_laya.evaluate(test_rows, eval_laya.rule_predictions(test_rows), "word-overlap rule (no model)")]
teachers = [json.load(open(f)) for f in sorted(glob.glob(str(WORK / "results" / "baseline_*.json")))]
logs = {n: json.load(open(WORK / "models" / n / "train_log.json")) for n in RUNS if (WORK / "models" / n / "train_log.json").exists()}
print("results:", list(res), "| teacher baselines:", [t["name"] for t in teachers])

## 8. Question 1: the learning curve
x = training rows (25 → 100 %), one line per score. The dashed line is the first run's soft_asis (4 epochs) for reference.
**Read it like this:** if the last segment (75 → 100 %) still rises by more than the noise (~1–2 points; one seed per point),
more data helps and generating it is worth it. If it is flat, the model has taken what this kind of data can give.

In [ ]:
import matplotlib.pyplot as plt
curve = [n for n in ("curve_25", "curve_50", "curve_75", "curve_100") if n in res]
x = [subsets[n]["rows"] for n in curve]

def score(n, group, key, part="verdict"):
    v = (res[n]["groups"].get(group) or {}).get(part) or {}
    s = v.get(key)
    return s[0] if isinstance(s, list) else s

panels = [("you", "accuracy", "you · accuracy"), ("you", "AUC P(not a quote)", "you · AUC"),
          ("you", "real quotes wrongly rejected", "you · real wrongly rejected"),
          ("you + Sonnet", "AUC P(not a quote)", "you + Sonnet · AUC"), ("you + Sonnet", "accuracy", "you + Sonnet · accuracy")]
fig, ax = plt.subplots(1, len(panels) + 1, figsize=(4 * (len(panels) + 1), 3.6))
for a, (g, k, title) in zip(ax, panels):
    a.plot(x, [score(n, g, k) for n in curve], marker="o")
    a.axhline(V1_SOFT_ASIS[title], ls="--", c="grey", lw=1, label="first run, soft_asis 4 ep")
    a.set(title=title, xlabel="training rows"); a.grid(alpha=.3)
best_val = [min(e["val"]["verdict"]["loss"] for e in logs[n]["epochs"]) if n in logs else None for n in curve]
ax[-1].plot(x, best_val, marker="o", c="C3"); ax[-1].set(title="val verdict loss (best epoch, lower = better)", xlabel="training rows")
ax[-1].grid(alpha=.3); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(WORK / "results" / "learning_curve.png", dpi=120); plt.show()

table = []
for n in curve:
    v = res[n]["groups"]["you"]["verdict"]; vs = res[n]["groups"]["you + Sonnet"]["verdict"]
    table.append({"run": n, "train rows": subsets[n]["rows"], "lectures": subsets[n]["lectures"],
                  "you · accuracy": eval_laya._fmt(v["accuracy"]), "you · AUC": v["AUC P(not a quote)"],
                  "you · real wrongly rejected": eval_laya._fmt(v["real quotes wrongly rejected"]),
                  "you · not-a-quote recall": eval_laya._fmt(v["not-a-quote recall"]),
                  "you+Sonnet · AUC": vs["AUC P(not a quote)"], "you+Sonnet · accuracy": eval_laya._fmt(vs["accuracy"])})
display(pd.DataFrame(table).set_index("run"))

## 9. Question 2: the boundary
**Balanced accuracy** is the column to read (mean recall of the 4 classes): "always complete" scores 0.25 whatever the class
mix, a model that catches cut-off quotes scores more. *Cut-off quotes caught* = of the quotes the labeller marked as cut off
(any `MISSING_*`), how many the model also flags as not complete: the part that matters for the app (it decides whether to
extend the span). The rule row is the bar: a model below it has not learned more than word matching.

On **your** cards the boundary is judged more strictly than Sonnet does (45 % complete vs 84–87 % on the same engines), so a
model trained on Sonnet's labels will under-flag cut-offs on your cards; *written text* is a second, independent judge.

In [ ]:
order = ["curve_100", "bnd_x3", "bnd_x6", "curve_25", "curve_50", "curve_75"]
models = [res[n] for n in order if n in res]
btext = []
for g in ["you", "written text", "you + Sonnet", "Sonnet · other lectures"]:
    t = f"### boundary · {g}\n" + eval_laya.compare_boundary(boundary_baselines + teachers + models, g)
    btext.append(t); display(Markdown(t))
(WORK / "results" / "boundary.md").write_text("\n\n".join(btext))

# val after each epoch: did the boundary start learning, and when?
display(pd.DataFrame([{"run": n, **{f"epoch {e['epoch']} · balanced": e["val"]["boundary"].get("balanced_accuracy") for e in logs[n]["epochs"]},
                       **{f"epoch {e['epoch']} · accuracy": round(e["val"]["boundary"]["accuracy"], 4) for e in logs[n]["epochs"]}}
                      for n in order if n in logs]).set_index("run"))

In [ ]:
# confusion matrices on your cards (rows = your label, columns = the model's answer)
for n in ("curve_100", "bnd_x3", "bnd_x6"):
    b = (res.get(n, {}).get("groups", {}).get("you") or {}).get("boundary")
    if b:
        display(Markdown(f"#### {n}"))
        display(pd.DataFrame(b["confusion (gold -> predicted)"]).T.rename_axis("you \\ model"))

## 10. Did the oversampling cost anything on the verdict?
The boundary repeats change what the shared encoder sees, so check the verdict did not get worse. Same columns as the first run.

In [ ]:
report = []
for g in ["you", "you + Sonnet", "Sonnet · other lectures", "written text"]:
    t = f"### verdict · {g}\n" + eval_laya.compare(models + teachers, g)
    report.append(t); display(Markdown(t))
(WORK / "results" / "comparison.md").write_text("\n\n".join(report))

for n in order:
    if n in logs:
        cal = logs[n].get("calibration", {})
        print(f"{n:10s} {logs[n]['seconds'] / 60:.0f} min | {logs[n]['train_items']} items ({logs[n].get('boundary_oversampled_items', 0)} boundary repeats) "
              f"| temperature {cal.get('temperature', ['?'])[0]} | issues {cal.get('report', {}).get('choice', {}).get('issues')}")

## 11. (Optional) Push the kept models to the Hugging Face Hub — private
Only the `KEEP_WEIGHTS` runs. Without `PUSH_TO_HUB`, download them from the version's **Output** tab (`models/<run>/`).

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    for n in RUNS:
        out = WORK / "models" / n
        if n not in KEEP_WEIGHTS or not (out / "model.safetensors").exists():
            continue
        repo = f"{HUB_PREFIX}-{n}"
        api.create_repo(repo, repo_type="model", private=True, exist_ok=True)
        api.upload_folder(folder_path=str(out), repo_id=repo, repo_type="model",
                          commit_message=f"laya-multilingual fine-tuned on Wahy laya_v1 ({n}, 2 epochs)")
        print("pushed", repo)